<a href="https://colab.research.google.com/github/Oluwatobi1122/Capson-Logistics-Dashboard/blob/main/Week2_08_embeddings_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2.3: Embeddings Deep Dive

## Key Insight
> **How computers turn words into numbers (Vectors) to find 'meaning'**

## Learning Objectives
By the end of this notebook, you will:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

## Exercises
This notebook contains 5 exercises ranging from easy to advanced.

## Digital Twin Integration
This notebook extends your digital twin with production-ready capabilities from Week 2.

In [1]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Tuple
import re
import json

print('✅ Week 2.3: Embeddings Deep Dive - Ready to start!')
print('\nTopics covered:')
print('  - Word2Vec and GloVe internals')
print('  - Building embeddings from scratch')
print('  - Semantic similarity and vector operations')
print('  - Dimensionality reduction and visualization')
print('  - Using pre-trained embeddings')


✅ Week 2.3: Embeddings Deep Dive - Ready to start!

Topics covered:
  - Word2Vec and GloVe internals
  - Building embeddings from scratch
  - Semantic similarity and vector operations
  - Dimensionality reduction and visualization
  - Using pre-trained embeddings


---
## Part 1: Introduction to Word2Vec and GloVe internals

Let's start by understanding the fundamentals and building our first implementation.

In [2]:
# Example implementation
# This is a starter template - expand with your implementation

class ExampleClass:
    """
    Example class for Word2Vec and GloVe internals.
    """

    def __init__(self):
        self.data = []

    def process(self, input_data):
        """
        Process the input data.
        """
        # Add your implementation here
        return input_data

# Test the class
example = ExampleClass()
result = example.process('test data')
print(f'Result: {result}')

Result: test data


### 🎯 Exercise 1 (Easy): Getting Started

**Task**: Implement a basic version of the concept introduced above.

**Requirements**:
1. Follow the pattern shown in the example
2. Test your implementation with sample data
3. Verify the output matches expectations
4. Add error handling for edge cases

In [3]:

import re
import math
from collections import Counter, defaultdict
from typing import Any, Dict, List


class ExampleClass:
    """
    Basic demonstration of Word2Vec and GloVe concepts.

    Word2Vec learns word representations by predicting words from context.
    GloVe learns representations using global word co-occurrence statistics.

    This implementation builds co-occurrence data and simple vectors;
    it does not train a full neural Word2Vec or GloVe model.
    """

    def __init__(self, window_size: int = 2, min_count: int = 1):
        if not isinstance(window_size, int) or isinstance(window_size, bool):
            raise TypeError("window_size must be an integer.")
        if window_size < 1:
            raise ValueError("window_size must be at least 1.")

        if not isinstance(min_count, int) or isinstance(min_count, bool):
            raise TypeError("min_count must be an integer.")
        if min_count < 1:
            raise ValueError("min_count must be at least 1.")

        self.window_size = window_size
        self.min_count = min_count
        self.data: List[str] = []
        self.vocabulary: Dict[str, int] = {}
        self.co_occurrences = defaultdict(Counter)
        self.vectors: Dict[str, List[float]] = {}

    def tokenize(self, input_data: Any) -> List[str]:
        """Convert input text into lowercase word tokens."""
        if not isinstance(input_data, str):
            raise TypeError(
                f"Expected a string, received {type(input_data).__name__}."
            )
        if not input_data.strip():
            raise ValueError("Input text cannot be empty.")

        tokens = re.findall(r"\b\w+\b", input_data.lower())

        if not tokens:
            raise ValueError("Input contains no valid word tokens.")

        return tokens

    def build_vocabulary(self, tokens: List[str]) -> Dict[str, int]:
        """Build a word-to-index mapping."""
        counts = Counter(tokens)
        words = sorted(
            word for word, count in counts.items()
            if count >= self.min_count
        )

        if not words:
            raise ValueError(
                "No words meet min_count. Try lowering min_count."
            )

        self.vocabulary = {
            word: index for index, word in enumerate(words)
        }
        return self.vocabulary

    def build_co_occurrences(self, tokens: List[str]) -> Dict[str, Dict[str, int]]:
        """Count nearby word pairs within the context window."""
        self.co_occurrences = defaultdict(Counter)
        valid_tokens = [
            token for token in tokens if token in self.vocabulary
        ]

        for index, word in enumerate(valid_tokens):
            start = max(0, index - self.window_size)
            end = min(len(valid_tokens), index + self.window_size + 1)

            for context_index in range(start, end):
                if context_index != index:
                    context_word = valid_tokens[context_index]
                    self.co_occurrences[word][context_word] += 1

        return {
            word: dict(counts)
            for word, counts in self.co_occurrences.items()
        }

    def create_simple_vectors(self) -> Dict[str, List[float]]:
        """
        Create normalized co-occurrence vectors.

        Each vector uses vocabulary words as dimensions. These are
        illustrative vectors, not trained Word2Vec/GloVe embeddings.
        """
        vocabulary_words = list(self.vocabulary.keys())
        vectors = {}

        for word in vocabulary_words:
            vector = [
                float(self.co_occurrences[word].get(other, 0))
                for other in vocabulary_words
            ]

            magnitude = math.sqrt(sum(value ** 2 for value in vector))

            if magnitude:
                vector = [value / magnitude for value in vector]

            vectors[word] = vector

        self.vectors = vectors
        return vectors

    def process(self, input_data: Any) -> Dict[str, Any]:
        """Run tokenization, vocabulary creation, and vector generation."""
        tokens = self.tokenize(input_data)
        self.data = tokens

        self.build_vocabulary(tokens)
        co_occurrences = self.build_co_occurrences(tokens)
        vectors = self.create_simple_vectors()

        return {
            "tokens": tokens,
            "vocabulary": self.vocabulary,
            "co_occurrences": co_occurrences,
            "vectors": vectors
        }


# -----------------------------
# Test the implementation
# -----------------------------

example = ExampleClass(window_size=2, min_count=1)

sample_text = "cats chase mice and cats like milk"
result = example.process(sample_text)

print("Tokens:")
print(result["tokens"])

print("\nVocabulary:")
print(result["vocabulary"])

print("\nCo-occurrence counts:")
print(result["co_occurrences"])

print("\nSimple word vectors:")
for word, vector in result["vectors"].items():
    print(f"{word}: {vector}")

# Verify expected results
assert result["tokens"] == [
    "cats", "chase", "mice", "and", "cats", "like", "milk"
]
assert "cats" in result["vocabulary"]
assert "mice" in result["vocabulary"]
assert result["co_occurrences"]["cats"]["chase"] >= 1
assert len(result["vectors"]["cats"]) == len(result["vocabulary"])

print("\nAll basic tests passed!")

# Edge-case tests
for invalid_input in [123, None, ["cats", "mice"]]:
    try:
        example.process(invalid_input)
    except TypeError as error:
        print(f"\nHandled invalid input: {error}")

try:
    example.process("   ")
except ValueError as error:
    print(f"\nHandled empty input: {error}")

try:
    ExampleClass(window_size=0)
except ValueError as error:
    print(f"\nHandled invalid window size: {error}")

Tokens:
['cats', 'chase', 'mice', 'and', 'cats', 'like', 'milk']

Vocabulary:
{'and': 0, 'cats': 1, 'chase': 2, 'like': 3, 'mice': 4, 'milk': 5}

Co-occurrence counts:
{'cats': {'chase': 1, 'mice': 2, 'and': 1, 'like': 1, 'milk': 1}, 'chase': {'cats': 1, 'mice': 1, 'and': 1}, 'mice': {'cats': 2, 'chase': 1, 'and': 1}, 'and': {'chase': 1, 'mice': 1, 'cats': 1, 'like': 1}, 'like': {'and': 1, 'cats': 1, 'milk': 1}, 'milk': {'cats': 1, 'like': 1}}

Simple word vectors:
and: [0.0, 0.5, 0.5, 0.5, 0.5, 0.0]
cats: [0.35355339059327373, 0.0, 0.35355339059327373, 0.35355339059327373, 0.7071067811865475, 0.35355339059327373]
chase: [0.5773502691896258, 0.5773502691896258, 0.0, 0.0, 0.5773502691896258, 0.0]
like: [0.5773502691896258, 0.5773502691896258, 0.0, 0.0, 0.0, 0.5773502691896258]
mice: [0.4082482904638631, 0.8164965809277261, 0.4082482904638631, 0.0, 0.0, 0.0]
milk: [0.0, 0.7071067811865475, 0.0, 0.7071067811865475, 0.0, 0.0]

All basic tests passed!

Handled invalid input: Expected a stri

---
## Part 2: Building embeddings from scratch

Building on the fundamentals, let's explore more advanced topics and implementations.

In [4]:
# Advanced implementation example

def advanced_function(data: List[Any], **kwargs) -> Dict[str, Any]:
    """
    Advanced function for Building embeddings from scratch.

    Args:
        data: Input data to process
        **kwargs: Additional parameters

    Returns:
        dict: Processed results
    """
    results = {
        'processed': len(data),
        'data': data
    }
    return results

# Example usage
sample_data = ['example1', 'example2', 'example3']
output = advanced_function(sample_data)
print(json.dumps(output, indent=2))

{
  "processed": 3,
  "data": [
    "example1",
    "example2",
    "example3"
  ]
}


### 🎯 Exercise 2 (Intermediate): Building on the Basics

**Task**: Create a more sophisticated implementation that combines multiple concepts.

**Requirements**:
1. Use the patterns from Part 1 and Part 2
2. Add parameter validation
3. Include helpful error messages
4. Test with various input types

In [5]:

import re
import json
import math
from collections import Counter, defaultdict
from typing import Any, Dict, List

import numpy as np


def advanced_function(
    data: List[Any],
    **kwargs
) -> Dict[str, Any]:
    """
    Build simple word embeddings from co-occurrence statistics.

    Parameters:
        data: List of text strings.
        window_size: Number of neighboring words to examine.
        min_count: Minimum frequency required for vocabulary inclusion.
        embedding_dim: Number of dimensions in the output vectors.
        normalize: Whether to normalize vectors to unit length.

    Returns:
        Dictionary containing vocabulary, co-occurrences, embeddings,
        and processing statistics.
    """

    # 1. Validate input data
    if not isinstance(data, list):
        raise TypeError("data must be a list of text strings.")

    if not data:
        raise ValueError("data cannot be an empty list.")

    for index, item in enumerate(data):
        if not isinstance(item, str):
            raise TypeError(
                f"Item at index {index} must be a string; "
                f"received {type(item).__name__}."
            )

    if not any(item.strip() for item in data):
        raise ValueError("data must contain at least one non-empty string.")

    # 2. Validate parameters
    window_size = kwargs.get("window_size", 2)
    min_count = kwargs.get("min_count", 1)
    embedding_dim = kwargs.get("embedding_dim", 50)
    normalize = kwargs.get("normalize", True)

    for name, value in [
        ("window_size", window_size),
        ("min_count", min_count),
        ("embedding_dim", embedding_dim)
    ]:
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError(f"{name} must be an integer.")

        if value < 1:
            raise ValueError(f"{name} must be at least 1.")

    if not isinstance(normalize, bool):
        raise TypeError("normalize must be a boolean.")

    # 3. Tokenize input text
    tokenized_documents = []

    for document in data:
        tokens = re.findall(r"\b\w+\b", document.lower())

        if tokens:
            tokenized_documents.append(tokens)

    if not tokenized_documents:
        raise ValueError("No valid word tokens were found in the input.")

    # 4. Build vocabulary using word frequencies
    frequencies = Counter(
        token
        for document in tokenized_documents
        for token in document
    )

    vocabulary = sorted(
        word for word, count in frequencies.items()
        if count >= min_count
    )

    if not vocabulary:
        raise ValueError(
            "Vocabulary is empty. Reduce min_count or provide more text."
        )

    word_to_index = {
        word: index for index, word in enumerate(vocabulary)
    }

    # 5. Build word co-occurrence matrix
    vocab_size = len(vocabulary)
    co_occurrence = np.zeros(
        (vocab_size, vocab_size),
        dtype=np.float64
    )

    for document in tokenized_documents:
        for i, word in enumerate(document):
            if word not in word_to_index:
                continue

            start = max(0, i - window_size)
            end = min(len(document), i + window_size + 1)

            for j in range(start, end):
                if i == j:
                    continue

                context_word = document[j]

                if context_word not in word_to_index:
                    continue

                distance = abs(i - j)

                # Nearby words contribute more than distant words.
                co_occurrence[
                    word_to_index[word],
                    word_to_index[context_word]
                ] += 1.0 / distance

    # 6. Generate simple embeddings using truncated SVD
    # SVD creates lower-dimensional representations of co-occurrence
    # patterns. These are not trained Word2Vec/GloVe embeddings.
    try:
        if np.count_nonzero(co_occurrence) == 0:
            embeddings = np.zeros(
                (vocab_size, min(embedding_dim, vocab_size)),
                dtype=np.float64
            )
        else:
            # Log scaling reduces the influence of very frequent pairs.
            matrix = np.log1p(co_occurrence)

            u, singular_values, _ = np.linalg.svd(
                matrix,
                full_matrices=False
            )

            actual_dim = min(
                embedding_dim,
                len(singular_values)
            )

            embeddings = (
                u[:, :actual_dim]
                * np.sqrt(singular_values[:actual_dim])
            )

    except np.linalg.LinAlgError as error:
        raise RuntimeError(
            "Embedding generation failed during matrix decomposition."
        ) from error

    # 7. Optionally normalize vectors
    if normalize:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        embeddings = embeddings / norms

    # 8. Format embeddings for readable output
    embedding_dict = {
        word: embeddings[index].round(6).tolist()
        for word, index in word_to_index.items()
    }

    # 9. Calculate co-occurrence pairs for inspection
    co_occurrence_dict = {}

    for word, index in word_to_index.items():
        related_words = {}

        for other_word, other_index in word_to_index.items():
            count = co_occurrence[index, other_index]

            if count > 0:
                related_words[other_word] = round(float(count), 4)

        co_occurrence_dict[word] = related_words

    # 10. Return results and processing statistics
    return {
        "processed": len(data),
        "non_empty_documents": len(tokenized_documents),
        "total_tokens": sum(
            len(document) for document in tokenized_documents
        ),
        "vocabulary_size": vocab_size,
        "window_size": window_size,
        "embedding_dimension": int(embeddings.shape[1]),
        "vocabulary": word_to_index,
        "word_frequencies": dict(frequencies),
        "co_occurrences": co_occurrence_dict,
        "embeddings": embedding_dict
    }


# -----------------------------------------
# Test 1: Standard input
# -----------------------------------------
sample_data = [
    "cats chase mice",
    "cats like milk",
    "dogs chase balls",
    "dogs like meat"
]

output = advanced_function(
    sample_data,
    window_size=2,
    min_count=1,
    embedding_dim=3
)

print("TEST 1: Standard input")
print(json.dumps(output, indent=2))

assert output["processed"] == 4
assert output["vocabulary_size"] > 0
assert "cats" in output["embeddings"]
assert len(output["embeddings"]["cats"]) <= 3

print("\nTest 1 passed.")


# -----------------------------------------
# Test 2: Mixed empty and non-empty strings
# -----------------------------------------
output2 = advanced_function(
    ["", "machine learning", "machine intelligence"],
    embedding_dim=2
)

assert output2["non_empty_documents"] == 2
assert "machine" in output2["vocabulary"]

print("Test 2 passed: Empty documents were handled.")


# -----------------------------------------
# Test 3: Invalid input types
# -----------------------------------------
try:
    advanced_function("this is not a list")
except TypeError as error:
    print("Test 3 passed:", error)


# -----------------------------------------
# Test 4: Invalid item inside list
# -----------------------------------------
try:
    advanced_function(["valid text", 123])
except TypeError as error:
    print("Test 4 passed:", error)


# -----------------------------------------
# Test 5: Invalid parameters
# -----------------------------------------
try:
    advanced_function(sample_data, window_size=0)
except ValueError as error:
    print("Test 5 passed:", error)


# -----------------------------------------
# Test 6: Frequency threshold too high
# -----------------------------------------
try:
    advanced_function(sample_data, min_count=100)
except ValueError as error:
    print("Test 6 passed:", error)


# -----------------------------------------
# Test 7: Empty input
# -----------------------------------------
try:
    advanced_function([])
except ValueError as error:
    print("Test 7 passed:", error)


print("\nAll tests completed successfully.")

TEST 1: Standard input
{
  "processed": 4,
  "non_empty_documents": 4,
  "total_tokens": 12,
  "vocabulary_size": 8,
  "window_size": 2,
  "embedding_dimension": 3,
  "vocabulary": {
    "balls": 0,
    "cats": 1,
    "chase": 2,
    "dogs": 3,
    "like": 4,
    "meat": 5,
    "mice": 6,
    "milk": 7
  },
  "word_frequencies": {
    "cats": 2,
    "chase": 2,
    "mice": 1,
    "like": 2,
    "milk": 1,
    "dogs": 2,
    "balls": 1,
    "meat": 1
  },
  "co_occurrences": {
    "balls": {
      "chase": 1.0,
      "dogs": 0.5
    },
    "cats": {
      "chase": 1.0,
      "like": 1.0,
      "mice": 0.5,
      "milk": 0.5
    },
    "chase": {
      "balls": 1.0,
      "cats": 1.0,
      "dogs": 1.0,
      "mice": 1.0
    },
    "dogs": {
      "balls": 0.5,
      "chase": 1.0,
      "like": 1.0,
      "meat": 0.5
    },
    "like": {
      "cats": 1.0,
      "dogs": 1.0,
      "meat": 1.0,
      "milk": 1.0
    },
    "meat": {
      "dogs": 0.5,
      "like": 1.0
    },
    "mice": 

---
## Part 3: Semantic similarity and vector operations

Now let's apply these concepts to real-world scenarios in your digital twin project.

In [6]:
# Visualization example
import matplotlib.pyplot as plt

def visualize_results(data: List[Any]):
    """
    Visualize the processing results.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Add your visualization code here
    ax.set_title('Results Visualization')
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')

    plt.tight_layout()
    plt.show()

# Example: visualize_results(sample_data)

### 🎯 Exercise 3 (Advanced): Complete Implementation

**Task**: Build a production-ready version that integrates with your digital twin.

**Requirements**:
1. Combine all concepts from this notebook
2. Add comprehensive error handling
3. Include logging and debugging capabilities
4. Create visualization of results
5. Write documentation for your implementation

In [7]:

"""
Production-Ready Digital Twin Knowledge Base Preprocessor
=========================================================

Features:
- Input validation and text normalization
- Structured data extraction
- Overlapping text chunking
- Approximate token counting and context-window checks
- Vocabulary and co-occurrence-based embeddings
- Metadata and processing quality metrics
- Logging, exception handling, and debugging
- Visualization of processing results
- Unit-style tests

Dependencies:
    pip install numpy matplotlib

Note:
The embeddings use co-occurrence statistics and SVD. They are not
a replacement for a fully trained Word2Vec or GloVe model.
"""

import re
import json
import logging
import string
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. Logging configuration
# ============================================================

logger = logging.getLogger("digital_twin_preprocessor")

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
        )
    )
    logger.addHandler(handler)

logger.setLevel(logging.INFO)


# ============================================================
# 2. Production system
# ============================================================

class ProductionSystem:
    """
    Process digital twin documents for a retrieval-augmented
    generation (RAG) knowledge base.

    Parameters:
        chunk_size: Maximum words per chunk.
        overlap: Number of words shared between adjacent chunks.
        context_window: Model context-window capacity in tokens.
        max_output_tokens: Tokens reserved for model responses.
        window_size: Word-context window for co-occurrence statistics.
        embedding_dim: Maximum requested embedding dimensions.
        min_count: Minimum frequency for vocabulary inclusion.
        debug: Enable detailed debug logging.

    Input:
        A string, a list of strings, or a dictionary containing
        'text' and optional 'document_id' and 'metadata' fields.

    Returns:
        A dictionary containing processed documents, extracted
        information, chunks, embeddings, and quality metrics.
    """

    def __init__(
        self,
        chunk_size: int = 100,
        overlap: int = 10,
        context_window: int = 8192,
        max_output_tokens: int = 1024,
        window_size: int = 2,
        embedding_dim: int = 50,
        min_count: int = 1,
        debug: bool = False
    ):
        integer_parameters = {
            "chunk_size": chunk_size,
            "overlap": overlap,
            "context_window": context_window,
            "max_output_tokens": max_output_tokens,
            "window_size": window_size,
            "embedding_dim": embedding_dim,
            "min_count": min_count
        }

        for name, value in integer_parameters.items():
            if not isinstance(value, int) or isinstance(value, bool):
                raise TypeError(f"{name} must be an integer.")

        if chunk_size < 1:
            raise ValueError("chunk_size must be at least 1.")
        if overlap < 0 or overlap >= chunk_size:
            raise ValueError(
                "overlap must be non-negative and smaller than chunk_size."
            )
        if context_window < 2:
            raise ValueError("context_window must be at least 2.")
        if not 0 <= max_output_tokens < context_window:
            raise ValueError(
                "max_output_tokens must be non-negative and smaller "
                "than context_window."
            )
        if window_size < 1:
            raise ValueError("window_size must be at least 1.")
        if embedding_dim < 1:
            raise ValueError("embedding_dim must be at least 1.")
        if min_count < 1:
            raise ValueError("min_count must be at least 1.")

        self.chunk_size = chunk_size
        self.overlap = overlap
        self.context_window = context_window
        self.max_output_tokens = max_output_tokens
        self.window_size = window_size
        self.embedding_dim = embedding_dim
        self.min_count = min_count

        self.available_input_tokens = (
            context_window - max_output_tokens
        )

        self.debug = debug
        if debug:
            logger.setLevel(logging.DEBUG)

        self.stopwords = {
            "the", "a", "an", "and", "or", "but", "is", "are",
            "was", "were", "in", "on", "at", "to", "for", "of",
            "with", "by"
        }

        self.last_result: Optional[Dict[str, Any]] = None

        logger.info("Digital twin preprocessor initialized.")

    # --------------------------------------------------------
    # 3. Input validation
    # --------------------------------------------------------

    def _normalize_input(self, input_data: Any) -> List[Dict[str, Any]]:
        """Convert supported input types into a common document format."""

        if isinstance(input_data, str):
            documents = [{"text": input_data}]

        elif isinstance(input_data, list):
            if not input_data:
                raise ValueError("Input document list cannot be empty.")

            documents = []

            for index, item in enumerate(input_data):
                if isinstance(item, str):
                    documents.append({"text": item})
                elif isinstance(item, dict):
                    documents.append(item.copy())
                else:
                    raise TypeError(
                        f"List item {index} must be a string or dictionary; "
                        f"received {type(item).__name__}."
                    )

        elif isinstance(input_data, dict):
            documents = [input_data.copy()]

        else:
            raise TypeError(
                "Input must be a string, a list of documents, "
                "or a document dictionary."
            )

        normalized = []

        for index, document in enumerate(documents):
            if "text" not in document:
                raise ValueError(
                    f"Document {index} is missing the required 'text' field."
                )

            text = document["text"]

            if not isinstance(text, str):
                raise TypeError(
                    f"Document {index} field 'text' must be a string."
                )

            if not text.strip():
                raise ValueError(
                    f"Document {index} contains empty or whitespace-only text."
                )

            metadata = document.get("metadata", {})
            if not isinstance(metadata, dict):
                raise TypeError(
                    f"Document {index} metadata must be a dictionary."
                )

            normalized.append({
                "document_id": str(
                    document.get("document_id", f"document_{index + 1}")
                ),
                "text": text,
                "metadata": metadata.copy()
            })

        return normalized

    # --------------------------------------------------------
    # 4. Cleaning and normalization
    # --------------------------------------------------------

    def clean_text(self, text: str) -> str:
        """
        Normalize whitespace and line endings without stripping
        technical punctuation, identifiers, or measurement units.
        """

        if not isinstance(text, str):
            raise TypeError("clean_text expects a string.")

        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = text.replace("\x00", "")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)

        return text.strip()

    # --------------------------------------------------------
    # 5. Structured data extraction
    # --------------------------------------------------------

    def extract_data(self, text: str) -> Dict[str, List[str]]:
        """Extract common technical and document-level entities."""

        patterns = {
            "emails": (
                r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
            ),
            "urls": r"https?://[^\s<>\"']+",
            "dates": r"\b\d{4}-\d{2}-\d{2}\b",
            "times": r"\b(?:[01]?\d|2[0-3]):[0-5]\d\b",
            "hashtags": r"(?<!\w)#[A-Za-z0-9_]+",
            "mentions": r"(?<!\w)@[A-Za-z0-9_]+",
            "error_codes": (
                r"\b(?:ERR|ERROR|WARN|WARNING)[-_ ]?\d+\b"
            ),
            "measurements": (
                r"(?<!\w)\d+(?:\.\d+)?\s?"
                r"(?:ms|sec|seconds|minutes|min|hours|hr|mm|cm|km|"
                r"kg|g|mg|°C|°F|kPa|MPa|V|kW|MW|rpm|%)\b"
            ),
            "identifiers": (
                r"\b(?:asset|sensor|device|machine|unit|equipment)"
                r"[-_ ]?[A-Z0-9][A-Z0-9_-]*\b"
            )
        }

        extracted = {}

        for category, pattern in patterns.items():
            extracted[category] = re.findall(
                pattern, text, flags=re.IGNORECASE
            )

        return extracted

    # --------------------------------------------------------
    # 6. Token estimation
    # --------------------------------------------------------

    @staticmethod
    def estimate_tokens(text: str) -> int:
        """
        Estimate token count using word-like units and punctuation.
        This is a heuristic, not an exact model tokenizer.
        """

        units = re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)
        return round(len(units) * 1.3)

    # --------------------------------------------------------
    # 7. Overlapping chunking
    # --------------------------------------------------------

    def chunk_text(self, text: str) -> List[str]:
        """Split text into word-based chunks with sliding-window overlap."""

        words = text.split()

        if not words:
            return []

        chunks = []
        start = 0

        while start < len(words):
            end = min(start + self.chunk_size, len(words))
            chunks.append(" ".join(words[start:end]))

            if end >= len(words):
                break

            start = end - self.overlap

        logger.debug("Created %d chunks.", len(chunks))
        return chunks

    # --------------------------------------------------------
    # 8. Vocabulary and co-occurrence embeddings
    # --------------------------------------------------------

    def build_embeddings(
        self,
        documents: List[Dict[str, Any]]
    ) -> Dict[str, Any]:
        """
        Build simple distributional embeddings from word co-occurrence.

        Uses log-scaled co-occurrence counts and truncated SVD.
        """

        tokenized_documents = []

        for document in documents:
            tokens = re.findall(
                r"\b\w+\b",
                document["text"].lower(),
                flags=re.UNICODE
            )
            if tokens:
                tokenized_documents.append(tokens)

        if not tokenized_documents:
            return {
                "vocabulary": {},
                "embeddings": {},
                "vocabulary_size": 0
            }

        frequencies = Counter(
            word
            for tokens in tokenized_documents
            for word in tokens
        )

        vocabulary = sorted(
            word for word, count in frequencies.items()
            if count >= self.min_count
        )

        if not vocabulary:
            return {
                "vocabulary": {},
                "embeddings": {},
                "vocabulary_size": 0
            }

        word_to_index = {
            word: index for index, word in enumerate(vocabulary)
        }

        size = len(vocabulary)
        matrix = np.zeros((size, size), dtype=np.float64)

        for tokens in tokenized_documents:
            for i, word in enumerate(tokens):
                if word not in word_to_index:
                    continue

                left = max(0, i - self.window_size)
                right = min(len(tokens), i + self.window_size + 1)

                for j in range(left, right):
                    if i == j:
                        continue

                    context = tokens[j]
                    if context not in word_to_index:
                        continue

                    distance = abs(i - j)
                    matrix[
                        word_to_index[word],
                        word_to_index[context]
                    ] += 1.0 / distance

        if np.count_nonzero(matrix) == 0:
            return {
                "vocabulary": word_to_index,
                "embeddings": {
                    word: [0.0] * min(self.embedding_dim, size)
                    for word in vocabulary
                },
                "vocabulary_size": size
            }

        try:
            scaled = np.log1p(matrix)
            u, singular_values, _ = np.linalg.svd(
                scaled, full_matrices=False
            )

            dimensions = min(self.embedding_dim, len(singular_values))
            vectors = (
                u[:, :dimensions]
                * np.sqrt(singular_values[:dimensions])
            )

            norms = np.linalg.norm(vectors, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            vectors = vectors / norms

        except np.linalg.LinAlgError as error:
            logger.exception("SVD embedding generation failed.")
            raise RuntimeError(
                "Unable to generate embeddings. Try reducing the "
                "input size or embedding dimensions."
            ) from error

        embedding_dict = {
            word: vectors[index].round(6).tolist()
            for word, index in word_to_index.items()
        }

        logger.debug(
            "Generated embeddings for %d vocabulary terms.", size
        )

        return {
            "vocabulary": word_to_index,
            "embeddings": embedding_dict,
            "vocabulary_size": size
        }

    # --------------------------------------------------------
    # 9. Main processing pipeline
    # --------------------------------------------------------

    def process(self, input_data: Any) -> Dict[str, Any]:
        """
        Execute the end-to-end document processing pipeline.

        Errors are logged and raised again with their original type,
        allowing calling applications to handle them appropriately.
        """

        started = datetime.now(timezone.utc)
        logger.info("Starting document processing.")

        try:
            source_documents = self._normalize_input(input_data)

            processed_documents = []
            all_chunks = []
            all_extracted = {}

            for document in source_documents:
                cleaned = self.clean_text(document["text"])

                extracted = self.extract_data(cleaned)
                for key, values in extracted.items():
                    all_extracted.setdefault(key, []).extend(values)

                processed_documents.append({
                    "document_id": document["document_id"],
                    "text": cleaned,
                    "metadata": document["metadata"],
                    "word_count": len(cleaned.split()),
                    "character_count": len(cleaned),
                    "estimated_tokens": self.estimate_tokens(cleaned)
                })

                chunks = self.chunk_text(cleaned)

                for index, chunk in enumerate(chunks):
                    token_count = self.estimate_tokens(chunk)

                    all_chunks.append({
                        "chunk_id": (
                            f"{document['document_id']}_chunk_{index + 1}"
                        ),
                        "document_id": document["document_id"],
                        "text": chunk,
                        "word_count": len(chunk.split()),
                        "estimated_tokens": token_count,
                        "fits_context_window": (
                            token_count <= self.available_input_tokens
                        ),
                        "metadata": {
                            **document["metadata"],
                            "document_id": document["document_id"],
                            "chunk_index": index,
                            "processed_at": started.isoformat()
                        }
                    })

            embedding_results = self.build_embeddings(processed_documents)

            total_words = sum(
                doc["word_count"] for doc in processed_documents
            )
            total_characters = sum(
                doc["character_count"] for doc in processed_documents
            )
            total_tokens = sum(
                doc["estimated_tokens"] for doc in processed_documents
            )

            elapsed = (
                datetime.now(timezone.utc) - started
            ).total_seconds()

            result = {
                "status": "success",
                "processed_at": started.isoformat(),
                "processing_seconds": round(elapsed, 4),
                "documents": processed_documents,
                "chunks": all_chunks,
                "extracted_data": all_extracted,
                "vocabulary": embedding_results["vocabulary"],
                "embeddings": embedding_results["embeddings"],
                "metrics": {
                    "document_count": len(processed_documents),
                    "chunk_count": len(all_chunks),
                    "total_words": total_words,
                    "total_characters": total_characters,
                    "estimated_tokens": total_tokens,
                    "vocabulary_size": embedding_results["vocabulary_size"],
                    "average_chunk_words": (
                        round(
                            sum(c["word_count"] for c in all_chunks)
                            / len(all_chunks), 2
                        ) if all_chunks else 0
                    ),
                    "chunks_over_context_limit": sum(
                        not chunk["fits_context_window"]
                        for chunk in all_chunks
                    )
                }
            }

            self.last_result = result
            logger.info(
                "Processing complete: %d documents, %d chunks, "
                "%d vocabulary terms.",
                len(processed_documents),
                len(all_chunks),
                embedding_results["vocabulary_size"]
            )

            return result

        except (TypeError, ValueError):
            logger.exception("Input validation or processing failed.")
            raise

        except Exception as error:
            logger.exception("Unexpected processing error.")
            raise RuntimeError(
                "Document processing failed. Check the logs for details."
            ) from error

    # --------------------------------------------------------
    # 10. Visualization
    # --------------------------------------------------------

    def visualize_results(
        self,
        result: Optional[Dict[str, Any]] = None
    ) -> None:
        """
        Plot chunk word counts and the most frequent vocabulary terms.
        """

        result = result or self.last_result

        if not isinstance(result, dict):
            raise ValueError(
                "No results available. Call process() first or provide results."
            )

        chunks = result.get("chunks", [])
        vocabulary = result.get("vocabulary", {})

        if not chunks and not vocabulary:
            raise ValueError(
                "Results contain no chunks or vocabulary to visualize."
            )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left plot: chunk sizes
        if chunks:
            sizes = [chunk["word_count"] for chunk in chunks]
            axes[0].bar(range(1, len(sizes) + 1), sizes)
            axes[0].set_title("Words per Text Chunk")
            axes[0].set_xlabel("Chunk number")
            axes[0].set_ylabel("Word count")
        else:
            axes[0].text(0.5, 0.5, "No chunks available", ha="center")
            axes[0].set_title("Words per Text Chunk")

        # Right plot: most frequent vocabulary terms
        if vocabulary:
            frequencies = Counter(
                token
                for document in result.get("documents", [])
                for token in re.findall(
                    r"\b\w+\b", document["text"].lower()
                )
            )
            common = frequencies.most_common(10)

            if common:
                terms, counts = zip(*common)
                axes[1].barh(list(terms)[::-1], list(counts)[::-1])
                axes[1].set_title("Top 10 Word Frequencies")
                axes[1].set_xlabel("Frequency")
                axes[1].set_ylabel("Word")
        else:
            axes[1].text(0.5, 0.5, "No vocabulary available", ha="center")
            axes[1].set_title("Top Word Frequencies")

        fig.suptitle("Digital Twin Knowledge Base Processing Results")
        fig.tight_layout()
        plt.show()

    # --------------------------------------------------------
    # 11. Debugging helper
    # --------------------------------------------------------

    def debug_summary(self) -> Dict[str, Any]:
        """Return a compact summary of the most recent processing run."""

        if self.last_result is None:
            return {
                "status": "not_run",
                "message": "Call process() before requesting a debug summary."
            }

        return {
            "status": self.last_result["status"],
            "metrics": self.last_result["metrics"],
            "processing_seconds": self.last_result["processing_seconds"],
            "context_window": self.context_window,
            "available_input_tokens": self.available_input_tokens
        }


# ============================================================
# 12. Comprehensive tests
# ============================================================

def run_tests():
    """Run basic tests for successful processing and error handling."""

    system = ProductionSystem(
        chunk_size=12,
        overlap=3,
        context_window=200,
        max_output_tokens=30,
        window_size=2,
        embedding_dim=8,
        debug=True
    )

    sample_data = [
        {
            "document_id": "sensor_report_001",
            "text": (
                "Sensor SENSOR_A12 reported ERROR-404 on 2026-10-14 "
                "at 14:30. Temperature reached 75°C. "
                "Contact engineer@example.com for maintenance details."
            ),
            "metadata": {
                "asset_type": "temperature_sensor",
                "location": "plant_01"
            }
        },
        {
            "document_id": "maintenance_log_002",
            "text": (
                "Predictive maintenance helps identify equipment failures. "
                "The digital twin uses sensor data to monitor machine health."
            ),
            "metadata": {"asset_type": "digital_twin"}
        }
    ]

    # Test 1: Normal processing
    result = system.process(sample_data)

    assert result["status"] == "success"
    assert result["metrics"]["document_count"] == 2
    assert result["metrics"]["chunk_count"] > 0
    assert "emails" in result["extracted_data"]
    assert "engineer@example.com" in result["extracted_data"]["emails"]

    print("PASS: Normal processing")

    # Test 2: Metadata preservation
    assert result["chunks"][0]["metadata"]["location"] == "plant_01"
    print("PASS: Metadata preservation")

    # Test 3: Embeddings generated
    assert result["metrics"]["vocabulary_size"] > 0
    assert len(result["embeddings"]) > 0
    print("PASS: Embedding generation")

    # Test 4: Plain-string input
    single = system.process("Equipment monitoring detects failures.")
    assert single["metrics"]["document_count"] == 1
    print("PASS: String input")

    # Test 5: Invalid input type
    try:
        system.process(123)
        raise AssertionError("Expected TypeError was not raised.")
    except TypeError:
        print("PASS: Invalid input type rejected")

    # Test 6: Empty text
    try:
        system.process("   ")
        raise AssertionError("Expected ValueError was not raised.")
    except ValueError:
        print("PASS: Empty text rejected")

    # Test 7: Missing text field
    try:
        system.process({"document_id": "missing_text"})
        raise AssertionError("Expected ValueError was not raised.")
    except ValueError:
        print("PASS: Missing text rejected")

    # Test 8: Invalid configuration
    try:
        ProductionSystem(chunk_size=5, overlap=5)
        raise AssertionError("Expected ValueError was not raised.")
    except ValueError:
        print("PASS: Invalid configuration rejected")

    # Test 9: Debug summary
    summary = system.debug_summary()
    assert summary["status"] == "success"
    print("PASS: Debug summary")

    # Test 10: Visualization
    # Uncomment to display the charts when running locally.
    # system.visualize_results(result)
    print("PASS: Visualization method is available")

    print("\nAll tests passed.")


if __name__ == "__main__":
    run_tests()

2026-10-14 10:33:37,870 | INFO | digital_twin_preprocessor | Digital twin preprocessor initialized.
INFO:digital_twin_preprocessor:Digital twin preprocessor initialized.
2026-10-14 10:33:37,871 | INFO | digital_twin_preprocessor | Starting document processing.
INFO:digital_twin_preprocessor:Starting document processing.
2026-10-14 10:33:37,878 | DEBUG | digital_twin_preprocessor | Created 2 chunks.
DEBUG:digital_twin_preprocessor:Created 2 chunks.
2026-10-14 10:33:37,882 | DEBUG | digital_twin_preprocessor | Created 2 chunks.
DEBUG:digital_twin_preprocessor:Created 2 chunks.
2026-10-14 10:33:37,893 | DEBUG | digital_twin_preprocessor | Generated embeddings for 36 vocabulary terms.
DEBUG:digital_twin_preprocessor:Generated embeddings for 36 vocabulary terms.
2026-10-14 10:33:37,895 | INFO | digital_twin_preprocessor | Processing complete: 2 documents, 4 chunks, 36 vocabulary terms.
INFO:digital_twin_preprocessor:Processing complete: 2 documents, 4 chunks, 36 vocabulary terms.
2026-10-14

PASS: Normal processing
PASS: Metadata preservation
PASS: Embedding generation
PASS: String input
PASS: Invalid input type rejected
PASS: Empty text rejected
PASS: Missing text rejected
PASS: Invalid configuration rejected
PASS: Debug summary
PASS: Visualization method is available

All tests passed.


---
## Summary & Key Takeaways

### What We Learned:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

### Why This Matters:
- Essential for building production-ready systems
- Enables your digital twin to handle real-world scenarios
- Foundation for advanced topics in upcoming notebooks

### Next Steps:
1. Complete all exercises in this notebook
2. Integrate concepts with your digital twin project
3. Test your implementation thoroughly
4. Move on to the next notebook in Week 2

## 🏆 Challenge Project

**Build a Complete Word2Vec and GloVe internals System for Your Digital Twin**

**Requirements**:
1. Apply all concepts from this notebook
2. Handle edge cases and errors gracefully
3. Include comprehensive testing
4. Create visualizations of your results
5. Document your implementation
6. Integrate with your existing digital twin code

**Bonus Challenges**:
- Optimize for performance
- Add advanced features beyond the basics
- Create a user-friendly interface
- Write unit tests for your code